1.Data Preparation

In [ ]:
# Import File
import pandas as pd
import numpy as np

file_path = "C:\\Users\\Acer\\OneDrive - Asia Pacific University\\Degree Y3S2\\FYP\\traffic_accidents.csv"
df = pd.read_csv(file_path)

In [ ]:
# dataset structure
df.info()

In [ ]:
# description statistic
df.describe()

In [ ]:
# Filter for Intersections ONLY
# check for missing values
df = df[df['intersection_related_i'] == 'Y'].copy()
df.isnull().sum()

In [ ]:
# Check UNKNOWN counts
unknown_cols = ['roadway_surface_cond', 'road_defect', 'traffic_control_device', 
                'weather_condition', 'alignment']
for col in unknown_cols:
    count = (df[col] == 'UNKNOWN').sum()
    print(f"{col}: {count}")

In [ ]:
# List the specific text values that act as "hidden" missing data
hidden_nulls = ['UNKNOWN', 'UNABLE TO DETERMINE']

# Count how many times these hidden nulls appear in each column
unknown_counts = df.isin(hidden_nulls).sum()

# Filter to only show columns that actually have at least one 'UNKNOWN'
# and sort them from highest count to lowest
unknown_counts_filtered = unknown_counts[unknown_counts > 0].sort_values(ascending=False)

print("Count of 'UNKNOWN' or 'UNABLE TO DETERMINE' per column:")
print("-" * 55)
# Print the results
if not unknown_counts_filtered.empty:
    print(unknown_counts_filtered)
else:
    print("No 'UNKNOWN' values found in any column!")

In [ ]:
# check duplicate records

num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# 1. Distribution of Crash Hour
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='crash_hour', color='skyblue')
plt.title('Distribution of Accidents by Hour of Day')
plt.xlabel('Hour (0-23)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 2. Distribution of Crash Day of Week
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='crash_day_of_week', color='salmon')
plt.title('Distribution of Accidents by Day of Week')
plt.xlabel('Day of Week (1=Sun, 7=Sat)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 3. Distribution of Crash Month
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='crash_month', color='lightgreen')
plt.title('Distribution of Accidents by Month')
plt.xlabel('Month')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 4. Damage Categories
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='damage', order=df['damage'].value_counts().index, palette='viridis')
plt.title('Frequency of Damage Categories')
plt.xlabel('Damage Amount')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 5. Weather Conditions (Top 10)
plt.figure(figsize=(10,6))
top_weather = df['weather_condition'].value_counts().head(10)
sns.barplot(x=top_weather.values, y=top_weather.index, palette='magma')
plt.title('Top 10 Weather Conditions')
plt.xlabel('Frequency')
plt.ylabel('Condition')
plt.show()

In [ ]:
# 6. Lighting Conditions
plt.figure(figsize=(10,6))
sns.countplot(data=df, y='lighting_condition',
              order=df['lighting_condition'].value_counts().index,
              palette='coolwarm')
plt.title('Distribution of Lighting Conditions')
plt.xlabel('Frequency')
plt.ylabel('Lighting Condition')
plt.show()

In [ ]:
# Injury distribution summary
injury_summary = df['injuries_total'].value_counts().head(5)

plt.figure(figsize=(8,5))
sns.barplot(x=injury_summary.index, y=injury_summary.values, palette="Blues_r")

plt.title("Top 5 Most Common Total Injury Counts")
plt.xlabel("Number of Injuries")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, y='most_severe_injury', order=df['most_severe_injury'].value_counts().index, palette='viridis')

plt.title('Distribution of Crash Severity', fontsize=14)
plt.xlabel('Number of Accidents', fontsize=12)
plt.ylabel('Most Severe Injury', fontsize=12)
plt.tight_layout()
plt.show()

# Print the exact percentages
print("Severity Percentages:")
print(df['most_severe_injury'].value_counts(normalize=True) * 100)

In [ ]:
# Correlation heatmap
numerical_df = df.select_dtypes(include=[np.number])

corr_matrix = numerical_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()

print(numerical_df.columns.tolist())
print(corr_matrix)

Data Preprocessing

In [ ]:
# Handle 'UNKNOWN' values
unknown_cols = [
    'weather_condition', 'lighting_condition', 'roadway_surface_cond', 
    'road_defect', 'traffic_control_device', 'alignment', 'trafficway_type'
]

for col in unknown_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()
    df[col] = df[col].replace('UNKNOWN', np.nan)

rows_before = df.shape[0]
df = df.dropna(subset=unknown_cols)
print(f"Rows removed due to UNKNOWN: {rows_before - df.shape[0]}")
print(f"Final shape: {df.shape}")

In [ ]:
# Remove duplicate records
print(f"\nShape before removing duplicates: {df.shape}")
df = df.drop_duplicates(keep='first')
print(f"Shape after removing duplicates: {df.shape}")

# Final verification
print(f"Duplicates remaining: {df.duplicated().sum()}")

In [ ]:
# Reset index
df = df.reset_index(drop=True)
print(f"Index reset complete.")
print(f"Total rows: {len(df)}")

In [ ]:
# Target Variable: Assign Risk Levels (Binary)
def assign_risk(row):
    # At Risk â€” any injury outcome or high-damage crash with injury
    if row['injuries_fatal'] > 0 or row['most_severe_injury'] == 'FATAL':
        return 1
    elif row['injuries_incapacitating'] > 0 or row['most_severe_injury'] == 'INCAPACITATING INJURY':
        return 1
    elif row['most_severe_injury'] in ['NONINCAPACITATING INJURY', 'REPORTED, NOT EVIDENT']:
        return 1
    elif row['damage'] == 'OVER $1,500' and row['injuries_total'] > 0:
        return 1
    # Low Risk
    else:
        return 0

df['Risk_Level'] = df.apply(assign_risk, axis=1)
print(df['Risk_Level'].value_counts())

In [ ]:
# Check class imbalance
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Count and percentage breakdown
risk_counts = df['Risk_Level'].value_counts().sort_index()
risk_pct    = df['Risk_Level'].value_counts(normalize=True).sort_index() * 100

risk_labels = {0: 'Low Risk', 1: 'At Risk'}
colors      = ['#5DCAA5', '#E24B4A']

print("Risk Level Distribution:")
print("-" * 40)
for level in [0, 1]:
    print(f"  {risk_labels[level]} (class {level}): "
          f"{risk_counts[level]:>6,}  ({risk_pct[level]:.1f}%)")
print(f"\n  Total records: {len(df):,}")

# Flag imbalance warning
dominant = risk_pct.max()
minority = risk_pct.min()
print("\nImbalance Assessment:")
if minority < 15:
    print(f"  âš  WARNING â€” Minority class is only {minority:.1f}%. "
          "Apply SMOTE or class_weight='balanced' during training.")
else:
    print(f"  âœ“ Class distribution is acceptable "
          f"(min class = {minority:.1f}%).")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar([risk_labels[i] for i in [0, 1]],
            [risk_counts[i] for i in [0, 1]],
            color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Risk Level â€” Record Count', fontsize=13)
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Number of Records')
for i, (cnt, pct) in enumerate(zip([risk_counts[j] for j in [0, 1]],
                                    [risk_pct[j]    for j in [0, 1]])):
    axes[0].text(i, cnt + risk_counts.max() * 0.01,
                 f'{cnt:,}\n({pct:.1f}%)', ha='center', fontsize=10)

# Pie chart
axes[1].pie([risk_counts[i] for i in [0, 1]],
            labels=[risk_labels[i] for i in [0, 1]],
            autopct='%1.1f%%', colors=colors,
            startangle=140, pctdistance=0.75)
axes[1].set_title('Risk Level â€” Class Proportion', fontsize=13)

plt.suptitle('Class Imbalance Analysis â€” Risk_Level (Binary)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Selected variables to PREDICT the target
selected_features = [
    # Environmental
    'weather_condition', 'lighting_condition', 'roadway_surface_cond',
    'road_defect', 'alignment', 'traffic_control_device', 'trafficway_type',
    # Temporal
    'crash_hour', 'crash_day_of_week', 'crash_month',
    'first_crash_type',   
    'num_units',    
]

X = df[selected_features]
y = df['Risk_Level']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures included:")
for f in selected_features:
    print(f"  {f}")

In [ ]:
from scipy.stats import chi2_contingency

# categorical environmental features (updated to include first_crash_type)
categorical_features = [
    'weather_condition', 'lighting_condition', 'roadway_surface_cond',
    'road_defect', 'alignment', 'traffic_control_device', 'trafficway_type',
    'first_crash_type'
]

target = 'Risk_Level'
chi_results = []

for col in categorical_features:
    contingency_table = pd.crosstab(df[col], df[target])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    chi_results.append({
        'Feature': col,
        'Chi-Square Statistic': round(chi2, 2),
        'P-Value': p,
        'Significant?': 'Yes' if p < 0.05 else 'No'
    })

chi_square_df = pd.DataFrame(chi_results)
chi_square_df = chi_square_df.sort_values(by='Chi-Square Statistic', ascending=False)

print("Chi-Square Test Results: Feature vs. Risk_Level")
print("-" * 65)
print(chi_square_df.to_string(index=False))



In [ ]:
# Statistical tests for numerical features
from scipy.stats import f_oneway, pointbiserialr
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Updated to include num_units
numerical_features = ['crash_hour', 'crash_day_of_week', 'crash_month', 'num_units']

print("One-Way ANOVA: Numerical Features vs. Risk_Level")
print("-" * 60)

anova_results = []
for col in numerical_features:
    groups = [df[df['Risk_Level'] == level][col].values for level in [0, 1]]
    f_stat, p_val = f_oneway(*groups)
    significant = 'Yes' if p_val < 0.05 else 'No'
    anova_results.append({
        'Feature': col,
        'F-Statistic': round(f_stat, 4),
        'P-Value': round(p_val, 6),
        'Significant?': significant
    })
    print(f"  {col:<22} F={f_stat:>8.4f}   p={p_val:.6f}   "
          f"Significant: {significant}")

anova_df = pd.DataFrame(anova_results)

# Point-Biserial Correlation
print("\nPoint-Biserial Correlation: Numerical Features vs. Each Risk Class")
print("-" * 60)
print(f"  {'Feature':<22} {'vs Low Risk (0)':>18} {'vs At Risk (1)':>18}")
print("  " + "-" * 58)

pb_results = []
for col in numerical_features:
    row = {'Feature': col}
    corrs = []
    for level in [0, 1]:
        binary_target = (df['Risk_Level'] == level).astype(int)
        r, p = pointbiserialr(df[col], binary_target)
        row[f'Class_{level}_r'] = round(r, 4)
        row[f'Class_{level}_p'] = round(p, 6)
        corrs.append(f"r={r:+.4f} (p={p:.4f})")
    pb_results.append(row)
    print(f"  {col:<22} {corrs[0]:>24} {corrs[1]:>24}")

pb_df = pd.DataFrame(pb_results)

# Visualise â€” boxplots of numerical features by Risk Level
fig, axes = plt.subplots(1, len(numerical_features), figsize=(5 * len(numerical_features), 5))
colors_box = {0: '#5DCAA5', 1: '#E24B4A'}
labels_box  = {0: 'Low Risk', 1: 'At Risk'}

for ax, col in zip(axes, numerical_features):
    data_by_class = [df[df['Risk_Level'] == level][col].values for level in [0, 1]]
    bp = ax.boxplot(data_by_class, patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2))
    for patch, level in zip(bp['boxes'], [0, 1]):
        patch.set_facecolor(colors_box[level])
        patch.set_alpha(0.85)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([labels_box[i] for i in [0, 1]])
    ax.set_title(f'{col}\nvs Risk Level', fontsize=12)
    ax.set_xlabel('Risk Level')
    ax.set_ylabel(col)

patches = [mpatches.Patch(color=colors_box[i], label=labels_box[i])
           for i in [0, 1]]
fig.legend(handles=patches, loc='upper right', fontsize=10)
plt.suptitle('Numerical Features Distribution by Risk Level (ANOVA)', fontsize=14)
plt.tight_layout()
plt.show()

Train/Test Split

In [ ]:

from sklearn.model_selection import train_test_split

# Split 1 â€” 80% train / 20% test
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Split 1 â€” 80/20")
print("-" * 50)
print(f"  X_train : {X_train_80.shape}")
print(f"  X_test  : {X_test_80.shape}")
print(f"  y_train : {y_train_80.shape}")
print(f"  y_test  : {y_test_80.shape}")

print("\n  Training set class distribution:")
for level, label in {0:'Low Risk', 1:'At Risk'}.items():
    cnt = (y_train_80 == level).sum()
    pct = cnt / len(y_train_80) * 100
    print(f"    {label} (class {level}): {cnt:>6,}  ({pct:.1f}%)")

print("\n  Test set class distribution:")
for level, label in {0:'Low Risk', 1:'At Risk'}.items():
    cnt = (y_test_80 == level).sum()
    pct = cnt / len(y_test_80) * 100
    print(f"    {label} (class {level}): {cnt:>6,}  ({pct:.1f}%)")

In [ ]:
# Split 2 â€” 70% train / 30% test
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Split 2 â€” 70/30")
print("-" * 50)
print(f"  X_train : {X_train_70.shape}")
print(f"  X_test  : {X_test_70.shape}")
print(f"  y_train : {y_train_70.shape}")
print(f"  y_test  : {y_test_70.shape}")

print("\n  Training set class distribution:")
for level, label in {0:'Low Risk', 1:'At Risk'}.items():
    cnt = (y_train_70 == level).sum()
    pct = cnt / len(y_train_70) * 100
    print(f"    {label} (class {level}): {cnt:>6,}  ({pct:.1f}%)")

print("\n  Test set class distribution:")
for level, label in {0:'Low Risk', 1:'At Risk'}.items():
    cnt = (y_test_70 == level).sum()
    pct = cnt / len(y_test_70) * 100
    print(f"    {label} (class {level}): {cnt:>6,}  ({pct:.1f}%)")

In [ ]:
#  summary
print("Split Comparison Summary")
print("=" * 55)
print(f"  {'Set':<18} {'80/20 Split':>16} {'70/30 Split':>16}")
print("  " + "-" * 52)
print(f"  {'X_train':<18} {str(X_train_80.shape):>16} {str(X_train_70.shape):>16}")
print(f"  {'X_test':<18} {str(X_test_80.shape):>16} {str(X_test_70.shape):>16}")
print(f"  {'Stratified':<18} {'Yes':>16} {'Yes':>16}")
print(f"  {'random_state':<18} {'42':>16} {'42':>16}")

Class Imbalance Handling (SMOTE)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import BorderlineSMOTE

# define columns (updated to include new features)
numerical_cols   = ['crash_hour', 'crash_day_of_week', 'crash_month', 'num_units']
categorical_cols = [
    'weather_condition', 'lighting_condition', 'roadway_surface_cond',
    'road_defect', 'alignment', 'traffic_control_device', 'trafficway_type',
    'first_crash_type'
]

# For Logistic Regression: Standard Scaler + One-Hot Encoding
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ])

# For Random Forest / Trees: Standard Scaler + Ordinal Encoding
preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ])

# ENCODE AND SCALE THE DATA
print("Transforming 80/20 split data...")
X_train_80_lr = preprocessor_lr.fit_transform(X_train_80)
X_test_80_lr  = preprocessor_lr.transform(X_test_80)

X_train_80_tree = preprocessor_tree.fit_transform(X_train_80)
X_test_80_tree  = preprocessor_tree.transform(X_test_80)

# Save raw (pre-SMOTE) tree arrays â€” needed by BalancedRandomForestClassifier
X_train_80_tree_raw = X_train_80_tree.copy()

# APPLY BorderlineSMOTE â€” focuses synthetic samples at the decision boundary
smote = BorderlineSMOTE(random_state=42, kind='borderline-1', k_neighbors=5)

print("Applying BorderlineSMOTE to training data...\n")
X_train_80_lr_bal,   y_train_80_bal = smote.fit_resample(X_train_80_lr,   y_train_80)
X_train_80_tree_bal, _              = smote.fit_resample(X_train_80_tree,  y_train_80)

print("-" * 50)
print("AFTER BorderlineSMOTE: 80% Training Set Class Distribution")
print("-" * 50)
for level, label in {0:'Low Risk', 1:'At Risk'}.items():
    cnt = (np.array(y_train_80_bal) == level).sum()
    pct = cnt / len(y_train_80_bal) * 100
    print(f"    {label} (class {level}): {cnt:>6,}  ({pct:.1f}%)")

print(f"\nFinal encoded matrix shape (LR)  : {X_train_80_lr_bal.shape}")
print(f"Final encoded matrix shape (Tree): {X_train_80_tree_bal.shape}")

In [ ]:
# â”€â”€ 70/30 split â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Re-initialise preprocessors â€” must refit on new training set
preprocessor_lr_70 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])
preprocessor_tree_70 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
])

print("Transforming 70/30 split data...")
X_train_70_lr   = preprocessor_lr_70.fit_transform(X_train_70)
X_test_70_lr    = preprocessor_lr_70.transform(X_test_70)
X_train_70_tree = preprocessor_tree_70.fit_transform(X_train_70)
X_test_70_tree  = preprocessor_tree_70.transform(X_test_70)

# Save raw (pre-SMOTE) tree arrays â€” needed by BalancedRandomForestClassifier
X_train_70_tree_raw = X_train_70_tree.copy()

print("Applying BorderlineSMOTE to training data...\n")
X_train_70_lr_bal,   y_train_70_bal = smote.fit_resample(X_train_70_lr,   y_train_70)
X_train_70_tree_bal, _              = smote.fit_resample(X_train_70_tree,  y_train_70)

print("-" * 50)
print("AFTER BorderlineSMOTE: 70% Training Set Class Distribution")
print("-" * 50)
for level, label in {0:'Low Risk', 1:'At Risk'}.items():
    cnt = (np.array(y_train_70_bal) == level).sum()
    pct = cnt / len(y_train_70_bal) * 100
    print(f"    {label} (class {level}): {cnt:>6,}  ({pct:.1f}%)")

print(f"\nFinal encoded matrix shape (LR)  : {X_train_70_lr_bal.shape}")
print(f"Final encoded matrix shape (Tree): {X_train_70_tree_bal.shape}")

In [ ]:
# Save all data for model training
import joblib

prepared_data = {
    # 80/20 split â€” SMOTE-balanced
    'X_train_80_lr'  : X_train_80_lr_bal,
    'X_train_80_tree': X_train_80_tree_bal,
    'X_test_80_lr'   : X_test_80_lr,
    'X_test_80_tree' : X_test_80_tree,
    'y_train_80'     : y_train_80_bal,
    'y_test_80'      : y_test_80,

    # 70/30 split â€” SMOTE-balanced
    'X_train_70_lr'  : X_train_70_lr_bal,
    'X_train_70_tree': X_train_70_tree_bal,
    'X_test_70_lr'   : X_test_70_lr,
    'X_test_70_tree' : X_test_70_tree,
    'y_train_70'     : y_train_70_bal,
    'y_test_70'      : y_test_70,

    # Raw (pre-SMOTE) tree arrays â€” for BalancedRandomForestClassifier
    'X_train_80_tree_raw': X_train_80_tree_raw,
    'y_train_80_raw'     : y_train_80,
    'X_train_70_tree_raw': X_train_70_tree_raw,
    'y_train_70_raw'     : y_train_70,

    # Preprocessors (needed for dashboard/inference)
    'preprocessor_lr'     : preprocessor_lr,
    'preprocessor_tree'   : preprocessor_tree,
    'preprocessor_lr_70'  : preprocessor_lr_70,
    'preprocessor_tree_70': preprocessor_tree_70,
}


print("Saved successfully â†’ fyp_preprocessed_data.pkl")
print("\nKeys saved:")
for k, v in prepared_data.items():
    shape = v.shape if hasattr(v, 'shape') else 'preprocessor object'
    print(f"  {k:<25} : {shape}")